# MyLLM - Complete Training Pipeline (Google Colab)
1. High-Speed Pretraining (140,000+ tokens/sec)
2. Supervised Instruction Fine-Tuning (SFT) for true Chatbot behavior
3. Interactive Web Chat Interface

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Pull latest training & SFT fine-tuning code
import os
if not os.path.exists('/content/myllm'):
    !git clone https://github.com/aaravsuthar45-bit/myllm.git /content/myllm
%cd /content/myllm
!git pull

In [ ]:
# 3. Install dependencies
%pip install -q tokenizers tqdm ftfy datasets flask

### 4. Sync Data from Google Drive (If applicable)

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/MY-LLM/data'
if os.path.exists(DRIVE_DATA):
    !rsync -av "$DRIVE_DATA/" /content/myllm/data/
    print("Synced data from Google Drive to local Colab SSD!")

### 5. Step 1: Pretrain Base Model (If not already trained)
Trains raw language fluency at 140k+ tokens/sec.

In [ ]:
!python training/pretrain.py \
    --batch_size 64 \
    --max_steps 5000 \
    --eval_every 250 \
    --log_every 50 \
    --learning_rate 4e-4

### 6. Step 2: Instruction Fine-Tuning (SFT) - The Secret to Chatbot Intelligence
Takes the pretrained model and teaches it to behave like ChatGPT by calculating loss strictly on Assistant responses.

In [ ]:
# 1. Prepare SFT conversation datasets with loss masks
!python training/prepare_sft_data.py \
    --input data/train.jsonl \
    --tokenizer data/tokenizer/tokenizer_combined.json \
    --output_dir data/processed \
    --max_seq_len 128

# 2. Fine-tune your pretrained weights into a conversational chatbot!
!python training/finetune_sft.py \
    --base_checkpoint checkpoints/pretrained_best.pt \
    --train_path data/processed/sft_train.pt \
    --val_path data/processed/sft_val.pt \
    --output_checkpoint checkpoints/sft_best.pt \
    --epochs 15 \
    --batch_size 16 \
    --learning_rate 1e-4

### 7. Download Best Chatbot Checkpoint to Laptop
Run this cell to save `sft_best.pt` directly to your laptop for local web chat!

In [ ]:
from google.colab import files
if os.path.exists('checkpoints/sft_best.pt'):
    files.download('checkpoints/sft_best.pt')
else:
    print("sft_best.pt not found yet. Run Step 6 first.")

### 8. Interactive Chatbot Web UI (In Colab)
Or chat with it directly inside Google Colab using the proxy URL.

In [ ]:
from google.colab.output import eval_js
print("Click this link to open your MyLLM Chatbot UI:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

!python chat_app.py --checkpoint checkpoints/sft_best.pt --port 5000